In [50]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.pipeline import Pipeline, make_pipeline

In [51]:
df = pd.read_csv("train.csv")
df.head()

# Step 1 -> train/test/split


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [52]:
df.drop(columns=["PassengerId", "Ticket", "Cabin", "Name"], inplace=True)
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['Survived']),
                                                 df['Survived'],
                                                 test_size=0.2,
                                                random_state=42)

In [53]:
trf1 = ColumnTransformer([
    ("imputer_age", SimpleImputer(), ["Age"]),
    ("impute_embarked", SimpleImputer(strategy="most_frequent"), ["Embarked"])
], remainder="passthrough", verbose_feature_names_out=False)
trf1.set_output(transform="pandas")

ColumnTransformer(remainder='passthrough',
                  transformers=[('imputer_age', SimpleImputer(), ['Age']),
                                ('impute_embarked',
                                 SimpleImputer(strategy='most_frequent'),
                                 ['Embarked'])],
                  verbose_feature_names_out=False)

In [54]:
trf2 = ColumnTransformer([
    ("ohe_sex_embarked", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["Sex", "Embarked"])
], remainder="passthrough")

In [55]:
trf3 = ColumnTransformer([
    ("scale", MinMaxScaler(), slice(0, 10))
])

In [56]:
trf4 = SelectKBest(
    score_func=chi2, k=8
)

In [57]:
trf5 = DecisionTreeClassifier()

In [58]:
pipe = Pipeline([
    ("trf1", trf1),
    ("trf2", trf2),
    ("trf3", trf3),
    ("trf4", trf4),
    ("trf5", trf5)
])

In [59]:
pipe.fit(X_train, y_train)

/Users/ayushthasale07/Sarg<3/PlainML/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/ayushthasale07/Sarg<3/PlainML/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/ayushthasale07/Sarg<3/PlainML/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/ayushthasale07/Sarg<3/PlainML/.venv/lib/python3.9/site-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behav

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('imputer_age',
                                                  SimpleImputer(), ['Age']),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  ['Embarked'])],
                                   verbose_feature_names_out=False)),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['Sex', 'Embarked'])])),
                ('trf3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('trf4',
                 SelectKBest(k=8, score_func=<function chi2 at 0x12bd7c430>)),
                ('trf5', DecisionTreeClassifier())])

In [60]:
pipe.named_steps["trf1"].transformers_[0][1].statistics_

array([29.49884615])

In [61]:
y_pred = pipe.predict(X_test)

In [62]:
from sklearn.metrics import accuracy_score

print(f"accuracy Score : {accuracy_score(y_pred, y_test)}")

accuracy Score : 0.7932960893854749
